# Enable Models as a Service (MaaS)

This notebook registers models with MaaS and configures access policies, API keys, and MCP gateway routes.

> **Note:** MaaS infrastructure (PostgreSQL, Gateway, Tenant) is pre-installed via [RHOAI-Toolkit](https://github.com/hyogrin/RHOAI-Toolkit). This notebook focuses on model registration, subscriptions, and API keys.

**What we'll do:**
1. **Verify MaaS infrastructure** (confirm gateway, tenant, and platform components)
2. Register model with MaaS (MaaSModelRef)
3. Configure access policy and subscription (MaaSAuthPolicy + MaaSSubscription)
4. Create API keys
5. List available models
6. Verify MCP server registrations
7. Verify the complete setup

**Prerequisites:**
- `0_setup/1_environment_setup.ipynb` completed (MaaS infra + model deployed)
- MCP servers deployed — see `1_mcp_servers/2_deploy_mcp_servers.ipynb`

**Reference:**
- [MaaS Official Docs](https://opendatahub-io.github.io/models-as-a-service/latest/)
- [MaaS Setup Guide](https://github.com/opendatahub-io/models-as-a-service/blob/main/docs/content/install/maas-setup.md)
- [Red Hat Connectivity Link 1.3](https://docs.redhat.com/en/documentation/red_hat_connectivity_link/1.3)
- [RHOAI 3.4 - Govern LLM access with MaaS](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.4/html/govern_llm_access_with_models-as-a-service/index)

## 0. Environment Setup

Loads environment variables from `.env`. Key variables:
- `CLUSTER_DOMAIN` — auto-detected if empty
- `MODEL_NAME`, `MODEL_NAMESPACE` — target model for registration

In [ ]:
import subprocess, json, os, time
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE")
MODEL_NAME = os.getenv("MODEL_NAME")

MAAS_HOST = f"https://maas-api.{CLUSTER_DOMAIN}"
INFERENCE_GW = os.getenv("MODEL_ENDPOINT", f"https://maas-api.{CLUSTER_DOMAIN}/{MODEL_NAMESPACE}/{MODEL_NAME}")

print(f"Cluster Domain:    {CLUSTER_DOMAIN}")
print(f"Model:             {MODEL_NAMESPACE}/{MODEL_NAME}")
print(f"MaaS API:          {MAAS_HOST}/maas-api/v1")
print(f"Inference Gateway: {INFERENCE_GW}")

Cluster Domain:    apps.openshift-cluster.sandbox2462.opentlc.com
Model:             /
MaaS API:          https://maas-api.apps.openshift-cluster.sandbox2462.opentlc.com/maas-api/v1
Inference Gateway: 


## 1. Verify MaaS Infrastructure

MaaS infrastructure is pre-installed via [RHOAI-Toolkit](https://github.com/hyogrin/RHOAI-Toolkit). Quick status check:

In [2]:
%%bash
source ../.env 2>/dev/null || true

APP_NS="redhat-ods-applications"
MAAS_NS="models-as-a-service"

echo "=== MaaS Gateway ==="
oc get gateway maas-default-gateway -n openshift-ingress 2>/dev/null || echo "(not found)"

echo ""
echo "=== Tenant ==="
oc get tenant -n ${MAAS_NS} 2>/dev/null || echo "(not found)"

echo ""
echo "=== MaaS API pods ==="
oc get pods -n ${APP_NS} -l app.kubernetes.io/name=maas-api --no-headers 2>/dev/null || echo "(not found)"

echo ""
echo "=== DSC modelsAsService ==="
DSC_NAME=$(oc get dsc -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)
echo "  $(oc get dsc ${DSC_NAME} -o jsonpath='{.spec.components.kserve.modelsAsService.managementState}' 2>/dev/null)"

echo ""
echo "✅ Proceed to Step 2."

=== MaaS Gateway ===
NAME                   CLASS                          ADDRESS                                                                  PROGRAMMED   AGE
maas-default-gateway   openshift-gateway-controller   acec56c4f76624083af3da2b3ba08658-850269517.us-east-2.elb.amazonaws.com   True         18h

=== Tenant ===
NAME             READY   REASON       AGE
default-tenant   True    Reconciled   9h

=== MaaS API pods ===
maas-api-75c8658794-txvw5   1/1   Running   0     6h2m

=== DSC modelsAsService ===
  Managed

✅ Proceed to Step 2.


## 2. Register Model with MaaS (MaaSModelRef)

Register the deployed model so it appears in the MaaS API (`/v1/models`).
The model can be running but invisible to MaaS without this step.

Ref: [Model Setup](https://opendatahub-io.github.io/models-as-a-service/latest/install/model-setup/)

In [3]:
%%bash
source ../.env 2>/dev/null || true
MODEL_NS=${MODEL_NAMESPACE:-demo}
MODEL=${MODEL_NAME:-qwen36-27b}

echo "=== 2.1 Verify LLMInferenceService exists ==="
oc get llminferenceservice ${MODEL} -n ${MODEL_NS} 2>/dev/null || {
    echo "ERROR: LLMInferenceService '${MODEL}' not found in namespace '${MODEL_NS}'."
    echo "Deploy it first: 0_setup/1_environment_setup.ipynb"
    exit 1
}

echo ""
echo "=== 2.2 Verify gateway ref on LLMInferenceService ==="
GW_REF=$(oc get llminferenceservice ${MODEL} -n ${MODEL_NS} -o jsonpath='{.spec.router.gateway.refs[0].name}' 2>/dev/null)
if [ "$GW_REF" = "maas-default-gateway" ]; then
    echo "Gateway ref: maas-default-gateway (correct)"
else
    echo "WARNING: Gateway ref is '${GW_REF}' — should be 'maas-default-gateway'"
    echo "Patch with: oc patch llminferenceservice ${MODEL} -n ${MODEL_NS} --type=json -p='[{\"op\":\"add\",\"path\":\"/spec/router/gateway/refs\",\"value\":[{\"name\":\"maas-default-gateway\",\"namespace\":\"openshift-ingress\"}]}]'"
fi

echo ""
echo "=== 2.3 Create/Update MaaSModelRef ==="
oc apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSModelRef
metadata:
  name: ${MODEL}
  namespace: ${MODEL_NS}
spec:
  modelRef:
    kind: LLMInferenceService
    name: ${MODEL}
EOF

echo ""
echo "Waiting for MaaSModelRef to reconcile..."
sleep 10

echo ""
echo "=== 2.4 Verify MaaSModelRef status ==="
PHASE=$(oc get maasmodelref ${MODEL} -n ${MODEL_NS} -o jsonpath='{.status.phase}' 2>/dev/null)
ENDPOINT=$(oc get maasmodelref ${MODEL} -n ${MODEL_NS} -o jsonpath='{.status.endpoint}' 2>/dev/null)
echo "Phase:    ${PHASE:-<pending>}"
echo "Endpoint: ${ENDPOINT:-<not set yet>}"

if [ "$PHASE" = "Ready" ]; then
    echo ""
    echo "✅ Model '${MODEL}' is registered with MaaS and ready."
else
    echo ""
    echo "⚠️  Model not yet Ready. Wait a minute and re-check:"
    echo "  oc get maasmodelref ${MODEL} -n ${MODEL_NS} -o yaml"
fi

=== 2.1 Verify LLMInferenceService exists ===
ERROR: LLMInferenceService 'qwen36-27b' not found in namespace 'demo'.
Deploy it first: 0_setup/1_environment_setup.ipynb


CalledProcessError: Command 'b'source ../.env 2>/dev/null || true\nMODEL_NS=${MODEL_NAMESPACE:-demo}\nMODEL=${MODEL_NAME:-qwen36-27b}\n\necho "=== 2.1 Verify LLMInferenceService exists ==="\noc get llminferenceservice ${MODEL} -n ${MODEL_NS} 2>/dev/null || {\n    echo "ERROR: LLMInferenceService \'${MODEL}\' not found in namespace \'${MODEL_NS}\'."\n    echo "Deploy it first: 0_setup/1_environment_setup.ipynb"\n    exit 1\n}\n\necho ""\necho "=== 2.2 Verify gateway ref on LLMInferenceService ==="\nGW_REF=$(oc get llminferenceservice ${MODEL} -n ${MODEL_NS} -o jsonpath=\'{.spec.router.gateway.refs[0].name}\' 2>/dev/null)\nif [ "$GW_REF" = "maas-default-gateway" ]; then\n    echo "Gateway ref: maas-default-gateway (correct)"\nelse\n    echo "WARNING: Gateway ref is \'${GW_REF}\' \xe2\x80\x94 should be \'maas-default-gateway\'"\n    echo "Patch with: oc patch llminferenceservice ${MODEL} -n ${MODEL_NS} --type=json -p=\'[{\\"op\\":\\"add\\",\\"path\\":\\"/spec/router/gateway/refs\\",\\"val

## 3. Configure Access Policy and Subscription

Create a **MaaSAuthPolicy** (who can access) and **MaaSSubscription** (rate limit quota).

> **Portal alternative:** You can also manage these via RHOAI Dashboard:
> *Models as a Service → Subscriptions* to create/edit subscriptions and policies through the UI.

Ref: [Quota and Access Configuration](https://opendatahub-io.github.io/models-as-a-service/latest/configuration-and-management/quota-and-access-configuration/)

In [ ]:
%%bash
source ../.env 2>/dev/null || true
MODEL_NS=${MODEL_NAMESPACE:-demo}
MODEL=${MODEL_NAME:-qwen36-27b}

echo "=== 3.1 Create group for lab users ==="
oc adm groups new lab-users 2>/dev/null || true
oc adm groups add-users lab-users $(oc whoami) 2>/dev/null || true
echo "Added $(oc whoami) to lab-users group"

echo ""
echo "=== 3.2 Create MaaSAuthPolicy ==="
oc apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSAuthPolicy
metadata:
  name: lab-access
  namespace: models-as-a-service
spec:
  modelRefs:
    - name: ${MODEL}
      namespace: ${MODEL_NS}
  subjects:
    groups:
      - name: lab-users
      - name: system:authenticated
    users: []
EOF

echo ""
echo "=== 3.3 Create MaaSSubscription (10000 tokens/min) ==="
oc apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSSubscription
metadata:
  name: lab-subscription
  namespace: models-as-a-service
spec:
  owner:
    groups:
      - name: lab-users
      - name: system:authenticated
    users: []
  modelRefs:
    - name: ${MODEL}
      namespace: ${MODEL_NS}
      tokenRateLimits:
        - limit: 10000
          window: 1m
  priority: 10
EOF

echo ""
echo "Waiting for policies to reconcile..."
sleep 15

echo ""
echo "=== 3.4 Verify generated policies ==="
echo "AuthPolicies:"
oc get authpolicy -n ${MODEL_NS} --no-headers 2>/dev/null || echo "  None"
echo "TokenRateLimitPolicies:"
oc get tokenratelimitpolicy -n ${MODEL_NS} --no-headers 2>/dev/null || echo "  None"

## 4. Create an API Key

![api key](https://raw.githubusercontent.com/hyogrin/rhoai-coding-assistant-lab/main/images/api_key.png)

API keys are the primary credential for accessing models and MCP tools through the MaaS gateway.

> **Portal alternative:** You can also create API keys via RHOAI Dashboard:
> *Models as a Service → API Keys* to generate and manage keys through the UI.

Ref: [API Key Management](https://opendatahub-io.github.io/models-as-a-service/latest/user-guide/api-key-management/)

In [ ]:
import urllib.request, ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

token_r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_r.stdout.strip()

key_data = json.dumps({
    "name": "lab-key",
    "description": "Key for code assistant lab",
    "expiresIn": "30d"
}).encode()

api_key_url = f"{MAAS_HOST}/maas-api/v1/api-keys"
req = urllib.request.Request(
    api_key_url, data=key_data,
    headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"},
    method="POST"
)

try:
    with urllib.request.urlopen(req, context=ctx, timeout=15) as resp:
        result = json.loads(resp.read())
    api_key = result.get("key", "")
    subscription = result.get("subscription", "")
    expires = result.get("expiresAt", "")

    if api_key and len(api_key) > 16:
        masked = api_key[:12] + "..." + api_key[-4:]
    else:
        masked = api_key

    env_path = os.path.join(os.path.dirname(os.path.abspath(".")), ".env")
    env_lines = []
    key_found = False
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.strip().startswith("MAAS_API_KEY="):
                    env_lines.append(f"MAAS_API_KEY={api_key}\n")
                    key_found = True
                else:
                    env_lines.append(line)
    if not key_found:
        env_lines.append(f"MAAS_API_KEY={api_key}\n")
    with open(env_path, "w") as f:
        f.writelines(env_lines)

    print(f"API Key created successfully!")
    print(f"  Key:          {masked}")
    print(f"  Subscription: {subscription}")
    print(f"  Expires:      {expires}")
    print(f"")
    print(f"  \u2705 Saved to .env as MAAS_API_KEY={masked}")
except urllib.error.HTTPError as e:
    body = e.read().decode() if e.fp else ""
    print(f"ERROR creating API key: HTTP {e.code}")
    print(f"  URL: {api_key_url}")
    print(f"  Response: {body[:300]}")
    if e.code == 404:
        print(f"  Hint: MaaS API may not be running. Check: oc get pods -l app.kubernetes.io/name=maas-api -A")
except Exception as e:
    print(f"ERROR: {e}")
    print(f"  URL attempted: {api_key_url}")

## 5. List Available Models via MaaS

In [ ]:
models_url = f"{MAAS_HOST}/maas-api/v1/models"
req = urllib.request.Request(
    models_url,
    headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"}
)

try:
    with urllib.request.urlopen(req, context=ctx, timeout=15) as resp:
        models_data = json.loads(resp.read())
    print("Models available via MaaS:")
    print("=" * 60)
    for model in models_data.get("data", []):
        print(f"  {model['id']:<30} url={model.get('url', 'N/A')}")
    if not models_data.get("data"):
        print("  No models found. Check MaaSModelRef status and MaaSAuthPolicy.")
except urllib.error.HTTPError as e:
    print(f"ERROR listing models: HTTP {e.code}")
    print(f"  Hint: Ensure MaaSAuthPolicy grants access to your group.")
except Exception as e:
    print(f"ERROR: {e}")

## 6. Verify MCP Server Registrations

MCP servers are registered with the MaaS gateway via HTTPRoute + AuthPolicy (created during MCP server deployment).
This step verifies the registrations exist and prints their endpoints.

In [ ]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_NS="mcp-servers"

echo "=== 6.1 HTTPRoutes ==="
ROUTES=$(oc get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true --no-headers 2>/dev/null)
if [ -n "$ROUTES" ]; then
    echo "$ROUTES"
else
    echo "No MCP HTTPRoutes found in namespace '${MCP_NS}'."
    echo "Deploy MCP servers first: 1_mcp_servers/2_deploy_mcp_servers.ipynb"
fi

echo ""
echo "=== 6.2 AuthPolicies ==="
oc get authpolicy -n ${MCP_NS} -l maas.opendatahub.io/managed=true --no-headers 2>/dev/null || echo "  None"

echo ""
echo "=== 6.3 MCP Endpoints via MaaS Gateway ==="
ROUTE_NAMES=$(oc get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{range .items[*]}{.metadata.name}{"\n"}{end}' 2>/dev/null)
if [ -n "$ROUTE_NAMES" ]; then
    for ROUTE in $ROUTE_NAMES; do
        PATH_PREFIX=$(oc get httproute ${ROUTE} -n ${MCP_NS} -o jsonpath='{.spec.rules[0].matches[0].path.value}' 2>/dev/null)
        echo "  https://maas-api.${CLUSTER_DOMAIN}${PATH_PREFIX}"
    done
    echo ""
    echo "✅ MCP server registrations verified."
else
    echo "  No routes found."
fi

## 7. Verify Complete Setup

In [ ]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MODEL_NS=${MODEL_NAMESPACE:-demo}
MODEL=${MODEL_NAME:-qwen36-27b}

echo "MaaS Setup Summary"
echo "============================================================"
echo ""
echo "Platform:"
echo "  Gateway:   $(oc get gateway maas-default-gateway -n openshift-ingress -o jsonpath='{.status.conditions[?(@.type=="Programmed")].status}' 2>/dev/null || echo 'N/A')"
echo "  MaaS API:  $(oc get pods -l app.kubernetes.io/name=maas-api -A --no-headers 2>/dev/null | wc -l | tr -d ' ') pod(s)"
echo "  RHCL:      $(oc get pods -n kuadrant-system --no-headers 2>/dev/null | wc -l | tr -d ' ') pod(s)"
echo ""
echo "Model Registration:"
echo "  MaaSModelRef:    $(oc get maasmodelref ${MODEL} -n ${MODEL_NS} -o jsonpath='{.status.phase}' 2>/dev/null || echo 'Not found')"
echo "  MaaSAuthPolicy:  $(oc get maasauthpolicy lab-access -n models-as-a-service -o jsonpath='{.metadata.name}' 2>/dev/null || echo 'Not found')"
echo "  MaaSSubscription:$(oc get maassubscription lab-subscription -n models-as-a-service -o jsonpath='{.metadata.name}' 2>/dev/null || echo 'Not found')"
echo ""
echo "MCP via Gateway:"
oc get httproute -n mcp-servers -l maas.opendatahub.io/managed=true --no-headers 2>/dev/null || echo "  No MCP routes"
echo ""
echo "Endpoints:"
echo "  Inference: https://maas-api.${CLUSTER_DOMAIN}/${MODEL_NS}/${MODEL}/v1"
echo "  MaaS API:  https://maas-api.${CLUSTER_DOMAIN}/maas-api/v1"
echo "  MCP Tools: https://maas-api.${CLUSTER_DOMAIN}/mcp/<server>/mcp"

## Summary

| Step | Resource | What It Does |
|------|----------|-------------|
| 1 | Infrastructure check | Verify MaaS gateway, DSC, tenant, DB secret |
| 2 | MaaSModelRef | Register model with MaaS catalog |
| 3 | MaaSAuthPolicy + MaaSSubscription | Grant access + define rate limits |
| 4 | API Key | Create credential for gateway access |
| 5 | List Models | Confirm model visible via MaaS API |
| 6 | Verify MCP registrations | Confirm MCP servers accessible through gateway |
| 7 | Verify | End-to-end status summary |

### Key MaaS CRDs

| CRD | Namespace | Purpose |
|-----|-----------|--------|
| `MaaSModelRef` | Model namespace | Register model for MaaS visibility |
| `MaaSAuthPolicy` | `models-as-a-service` | Who can access which models |
| `MaaSSubscription` | `models-as-a-service` | Token rate limits per group |
| `Tenant` | `models-as-a-service` | Platform config (gateway ref, telemetry) |

## Next Steps

- `3_test_model_serving.ipynb` — Test inference, streaming, and auth through gateway
- `4_test_mcp_servers.ipynb` — Test MCP server access through gateway
- `../4_control/` — Rate limit testing and multi-tier subscription demo